In [1]:
import mysql.connector
import pandas as pd
import numpy as np

In [2]:
conn = mysql.connector.connect(
    host = 'localhost',
    user = 'root',
    password = 'pass@123',
    database = 'ecom'
    
)


products_df = pd.read_csv("D:\\Works\\ETL_Project\\products_data.csv")

users_df = pd.read_csv("D:\\Works\\ETL_Project\\users_data.csv")

orders_df = pd.read_csv("D:\\Works\\ETL_Project\\orders_data.csv")

payments_df = pd.read_excel("D:\\Works\\ETL_Project\\payments_data.xlsx", engine = 'openpyxl')

review_df = pd.read_excel("D:\\Works\\ETL_Project\\reviews_data.xlsx" ,engine = 'openpyxl')


print("Success!")

Success!


In [3]:
# ============================================================
# PRODUCTS TABLE
# ============================================================

products_df = products_df.fillna(
    {
        'Description': 'No Description',
        'Category': 'Miscellaneous'
    }
)

# Convert Price to numeric
products_df['Price'] = (
    pd.to_numeric(products_df['Price'], errors='coerce')
    .fillna(0.0)
    .round(2)
)

# to_numeric - converts the data to numeric datatype
# errors='coerce' - values that cannot be converted become NaN
# fillna(0.0) - replaces NaN values with 0.0
# round(2) - rounds the value to 2 decimal places


# Convert StockQuantity to integer
products_df['StockQuantity'] = (
    pd.to_numeric(products_df['StockQuantity'], errors='coerce')
    .fillna(0)
    .astype(int)
)

# to_numeric - converts StockQuantity to numeric
# errors='coerce' - invalid values become NaN
# fillna(0) - replaces NaN with 0
# astype(int) - converts values to integer


# ============================================================
# USERS TABLE
# ============================================================

def generate_unique_email(index):
    return f'unknown{index}@example.com'


users_df['Email'] = users_df.apply(
    lambda row: (
        row['Email']
        if isinstance(row['Email'], str)
        and '@' in row['Email']
        and '.' in row['Email']
        else generate_unique_email(row.name + 1)
    ),
    axis=1
)

# row - represents one row of the DataFrame
# isinstance(..., str) - checks whether Email is a string
# '@' in row['Email'] - checks whether email contains @
# '.' in row['Email'] - checks whether email contains .
# If email is invalid, generate_unique_email() creates a new email
# axis=1 - applies the function row by row


# ============================================================
# ORDERS TABLE
# ============================================================

orders_df = orders_df.fillna(
    {
        'TotalAmount': 0.0
    }
)

# Remove extra spaces from OrderDate
orders_df['OrderDate'] = orders_df['OrderDate'].astype(str).str.strip()

# strip() - removes extra spaces from beginning and end


# Convert OrderDate to datetime
orders_df['OrderDate'] = pd.to_datetime(
    orders_df['OrderDate'],
    errors='coerce'
)

# errors='coerce' - invalid dates become NaT


# Replace invalid/missing dates with default date
orders_df['OrderDate'] = orders_df['OrderDate'].fillna(
    pd.Timestamp('1970-01-01')
)

# fillna() - replaces missing values
# pd.Timestamp() - creates a pandas datetime value


# Convert date into YYYY-MM-DD format
orders_df['OrderDate'] = orders_df['OrderDate'].dt.strftime(
    '%Y-%m-%d'
)

# strftime() - converts datetime into the required string format


# Create original Order ID for mapping
orders_df['OldOrderID'] = range(
    1001,
    1001 + len(orders_df)
)


# ============================================================
# PAYMENTS TABLE
# ============================================================

payments_df = payments_df.fillna(
    {
        'Amount': 0.0
    }
)

# Convert PaymentDate to datetime
payments_df['PaymentDate'] = pd.to_datetime(
    payments_df['PaymentDate'],
    errors='coerce',
    format='%Y-%m-%d'
)

# Invalid dates become NaT


# Replace invalid/missing dates with default date
payments_df['PaymentDate'] = payments_df['PaymentDate'].fillna(
    pd.Timestamp('1970-01-01')
)


# Convert PaymentDate to YYYY-MM-DD format
payments_df['PaymentDate'] = payments_df['PaymentDate'].dt.strftime(
    '%Y-%m-%d'
)


# Replace invalid payment method
payments_df['PaymentMethod'] = payments_df['PaymentMethod'].replace(
    'InvalidMethod',
    'Unknown'
)

# replace() - replaces old value with new value


# ============================================================
# REVIEW TABLE
# ============================================================

review_df = review_df.fillna(
    {
        'ReviewText': 'No Review Available'
    }
)

# Replace InvalidRating with 0
review_df['Rating'] = review_df['Rating'].replace(
    'InvalidRating',
    0
)

# Convert Rating to numeric
review_df['Rating'] = pd.to_numeric(
    review_df['Rating'],
    errors='coerce'
).fillna(0).astype(int)


# ============================================================
# DISPLAY TRANSFORMED DATA
# ============================================================

print(products_df)
print(users_df)
print(orders_df)
print(payments_df)
print(review_df)

print('Transformed Successfully....')

      ProductName                  Description   Price  StockQuantity  \
0   Contraption A               No Description    0.00              0   
1          Tool B         A high-quality tool.  230.23            160   
2       Machine C    A high-quality apparatus.   56.75             96   
3        Device D               No Description  191.70            472   
4   Contraption E       A high-quality gadget.  337.73            233   
5     Appliance F    A high-quality appliance.  336.30            180   
6         Gizmo G               No Description  299.74            113   
7   Contraption H       A high-quality gadget.  144.61            318   
8       Machine I    A high-quality appliance.  285.01              0   
9        Device J               No Description  197.63            497   
10         Tool K         A high-quality tool.    0.00            442   
11      Machine L      A high-quality machine.  486.14             52   
12      Machine M               No Description  425

C:\Users\Admin\AppData\Local\Temp\ipykernel_4028\4154396932.py:82: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  orders_df['OrderDate'] = pd.to_datetime(


In [7]:
# load

cursor = conn.cursor()
# cursor - databse handle - to execute sql command in database
 
# delete previos records - to avoid duplication
table_to_clear = ['reviews','payments','orders','users','products']
for table in table_to_clear:
    cursor.execute(f'delete from {table}') # delete command in a loop
conn.commit()

product_id_mapping = {} # empty dictionary to store product ids
# stores new product_id assigned by system
for index,row in products_df.iterrows():
    # index -holds index number of each row
    # interrow - will loop through each row
    cursor.execute('''insert into products(product_name,p_description,price,stock_quantity,category) values (%s,%s,%s,%s,%s)''',
                  (row['ProductName'],row['Description'],row['Price'],row['StockQuantity'],row['Category']))
    product_id_mapping[index +1] = cursor.lastrowid
    # index+1 -key index +1 (ex - 0+1 -1)
    # lastrowid - return the last id added (auto_increment value) - value
    
conn.commit()
print(product_id_mapping)


user_id_mapping = {}
for index,row in users_df.iterrows():
    cursor.execute('''insert into users(username,email,address,u_password) values (%s,%s,%s,%s)''',
                  (row['UserName'],row['Email'],row['Address'],row['Password']))
    user_id_mapping[index +1]= cursor.lastrowid
conn.commit()
print(user_id_mapping)


# update orders_df with new user id
orders_df['UserID'] = orders_df['UserID'].map(user_id_mapping)
#orders_df['UserID'] = orders_df.index.map(user_id_mapping)
#map() - takes dictionary as argument.it will map each key to the respective
print(orders_df['UserID'])
orders_df = orders_df.dropna(subset = ['UserID'])
#some user_ids may be missing , so NULL
# dropna - will drp the row where userID is null


order_id_mapping = {}
for index,row in orders_df.iterrows():
    cursor.execute('''insert into orders(userid,order_date,total_amt,email) values (%s,%s,%s,%s)''',(int(row['UserID']),row['OrderDate'],
                                                                                                    row['TotalAmount'],row['Email']))
    order_id_mapping[row['OldOrderID']] = cursor.lastrowid
conn.commit()
print(order_id_mapping)


payments_df['OrderID'] = payments_df['OrderID'].map(order_id_mapping)
print(payments_df['OrderID'])
payments_df = payments_df.dropna(subset = ['OrderID'])


for index,row in payments_df.iterrows():
    cursor.execute('''insert into payments (orderid,payment_method,payment_date,amount) values (%s,%s,%s,%s)''',(int(row['OrderID']),
                                                                                                                row['PaymentMethod'],row['PaymentDate'],row['Amount']))
conn.commit()






review_df['UserID'] = review_df['UserID'].map(user_id_mapping)
review_df['ProductID'] = review_df['ProductID'].map(product_id_mapping)
review_df = review_df.dropna(subset=['ProductID','UserID'])
for index,row  in review_df.iterrows():
    product_id = product_id_mapping.get(index)
    user_id = user_id_mapping.get(index)
    cursor.execute('''insert into reviews (productid,userid,rating,review_text) values (%s,%s,%s,%s)''',(int(row['ProductID']),int(row['UserID']),int(row['Rating']),row['ReviewText']))
conn.commit()
print('Loaded Successfully....')
#cursor.close()
    


{1: 181, 2: 182, 3: 183, 4: 184, 5: 185, 6: 186, 7: 187, 8: 188, 9: 189, 10: 190, 11: 191, 12: 192, 13: 193, 14: 194, 15: 195, 16: 196, 17: 197, 18: 198, 19: 199, 20: 200, 21: 201, 22: 202, 23: 203, 24: 204, 25: 205, 26: 206, 27: 207, 28: 208, 29: 209, 30: 210, 31: 211, 32: 212, 33: 213, 34: 214, 35: 215, 36: 216, 37: 217, 38: 218, 39: 219, 40: 220, 41: 221, 42: 222, 43: 223, 44: 224, 45: 225, 46: 226, 47: 227, 48: 228, 49: 229, 50: 230, 51: 231, 52: 232, 53: 233, 54: 234, 55: 235, 56: 236, 57: 237, 58: 238, 59: 239, 60: 240}
{1: 11, 2: 12, 3: 13, 4: 14, 5: 15, 6: 16, 7: 17, 8: 18, 9: 19, 10: 20}
0     18
1     14
2     11
3     18
4     14
5     16
6     18
7     14
8     13
9     19
10    13
11    19
12    12
13    12
14    12
15    16
16    13
17    19
18    14
19    11
20    14
21    11
22    15
23    14
24    18
25    18
26    17
27    13
28    11
29    11
30    13
31    16
32    17
33    16
34    16
35    16
36    13
37    16
38    18
39    12
40    15
41    11
42    11
43    15
